In [6]:
from snowflake.snowpark import Session

conn_parameters = {
    "account": "EVFUUFU-KN22973",
    "user": "AMMU",
    "password": "aammeeyyAAMMEEYY7763",
    "role": "ACCOUNTADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "AVD",
    "schema": "TPCH_SF1"
}   

session = Session.builder.configs(conn_parameters).create()
result = session.sql("SELECT CURRENT_warehouse(), CURRENT_DATABASE()").collect()
print(result)


from snowflake.snowpark.functions import col
from datetime import date
from snowflake.snowpark.types import StructType, StructField
from snowflake.snowpark.types import IntegerType, StringType, DateType

[Row(CURRENT_WAREHOUSE()='COMPUTE_WH', CURRENT_DATABASE()='AVD')]


In [7]:
import pandas as pd
from snowflake.snowpark import Session

In [8]:
print("Pandas version:", pd.__version__)
df = pd.read_csv(r'C:\Users\Amey\Downloads\orders (1).csv')
df.head()


Pandas version: 2.3.3


,ORDER_ID,AMOUNT,PROFIT,QUANTITY,CATEGORY,SUBCATEGORY
0,B-25601,1275,-1148,7,Furniture,Bookcases
1,B-25601,66,-12,5,Clothing,Stole
2,B-25601,8,-2,3,Clothing,Hankerchief
3,B-25601,80,-56,4,Electronics,Electronic Games
4,B-25602,168,-111,2,Electronics,Phones


In [10]:
session.sql("USE DATABASE AVD").collect()
session.sql("USE SCHEMA TEST").collect()

snowpark_df = session.create_dataframe(df)
snowpark_df.show()

----------------------------------------------------------------------------------
|"ORDER_ID"  |"AMOUNT"  |"PROFIT"  |"QUANTITY"  |"CATEGORY"   |"SUBCATEGORY"     |
----------------------------------------------------------------------------------
|B-25601     |1275      |-1148     |7           |Furniture    |Bookcases         |
|B-25601     |66        |-12       |5           |Clothing     |Stole             |
|B-25601     |8         |-2        |3           |Clothing     |Hankerchief       |
|B-25601     |80        |-56       |4           |Electronics  |Electronic Games  |
|B-25602     |168       |-111      |2           |Electronics  |Phones            |
|B-25602     |424       |-272      |5           |Electronics  |Phones            |
|B-25602     |2617      |1151      |4           |Electronics  |Phones            |
|B-25602     |561       |212       |3           |Clothing     |Saree             |
|B-25602     |119       |-5        |8           |Clothing     |Saree             |
|B-2

In [11]:
session.sql("""
SELECT CURRENT_DATABASE(), CURRENT_SCHEMA()
""").show()

---------------------------------------------
|"CURRENT_DATABASE()"  |"CURRENT_SCHEMA()"  |
---------------------------------------------
|AVD                   |TEST                |
---------------------------------------------



In [22]:
session.sql("SHOW SCHEMAS IN DATABASE AVD").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"              |"is_default"  |"is_current"  |"database_name"  |"owner"       |"comment"                                           |"options"  |"retention_time"  |"owner_role_type"  |"classification_profile_database"  |"classification_profile_schema"  |"classification_profile"  |"object_visibility"  |"is_nested"  |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
session.sql("SHOW SCHEMAS IN DATABASE AVD").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"              |"is_default"  |"is_current"  |"database_name"  |"owner"       |"comment"                                           |"options"  |"retention_time"  |"owner_role_type"  |"classification_profile_database"  |"classification_profile_schema"  |"classification_profile"  |"object_visibility"  |"is_nested"  |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
session.sql("SHOW STAGES IN SCHEMA AVD.TEST").show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"created_on"                      |"name"                          |"database_name"  |"schema_name"  |"url"  |"has_credentials"  |"has_encryption_key"  |"owner"       |"comment"  |"region"  |"type"              |"cloud"  |"notification_channel"  |"storage_integration"  |"endpoint"  |"owner_role_type"  |"directory_enabled"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-09-09 04:0

In [36]:
obj = session.file.put(
    r"C:\Users\Amey\Downloads\orders.avro",
    "@AVD.TEST.STAGES"
)
print(obj)


[PutResult(source='orders.avro', target='orders.avro.gz', source_size=46000, target_size=0, source_compression='NONE', target_compression='GZIP', status='SKIPPED', message='')]


In [ ]:
SNOWFLAKE
                       │
                       ▼
                      AVD
                       │
                       ▼
                      TEST
                 ┌─────┴─────┐
                 │           │
                 ▼           ▼
              STAGES    Snowpark temp
              (your       stages
               stage)
                 │
                 ▼
           orders.avro

In [53]:
session.sql("""
CREATE OR REPLACE FILE FORMAT AVD.TEST.CSV_FORMAT
TYPE = CSV
SKIP_HEADER = 1
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
COMPRESSION = AUTO
""").collect()

[Row(status='File format CSV_FORMAT successfully created.')]

In [37]:
session.sql("LIST @AVD.TEST.STAGES").show()

----------------------------------------------------------------------------------------------------
|"name"                 |"size"  |"md5"                             |"last_modified"               |
----------------------------------------------------------------------------------------------------
|stages/orders.avro.gz  |10128   |7fd644c69511ed15656bbb530f49814c  |Wed, 9 Sep 2026 11:51:24 GMT  |
----------------------------------------------------------------------------------------------------



In [38]:
session.sql("""
CREATE OR REPLACE FILE FORMAT AVD.TEST.AVRO_FORMAT
TYPE = AVRO
COMPRESSION = AUTO
""").collect()

[Row(status='File format AVRO_FORMAT successfully created.')]

In [42]:
session.file.put(r"C:\Users\Amey\Downloads\customer.csv","@AVD.TEST.STAGES")

[PutResult(source='customer.csv', target='customer.csv.gz', source_size=815016, target_size=303872, source_compression='NONE', target_compression='GZIP', status='UPLOADED', message='')]

In [45]:
session.sql("LIST @AVD.TEST.STAGES").show()

-----------------------------------------------------------------------------------------------------
|"name"                  |"size"  |"md5"                             |"last_modified"               |
-----------------------------------------------------------------------------------------------------
|stages/customer.csv.gz  |303872  |d9c1e2a5cc11b484b6727aad02f174ff  |Wed, 9 Sep 2026 12:04:58 GMT  |
|stages/orders.avro.gz   |10128   |7fd644c69511ed15656bbb530f49814c  |Wed, 9 Sep 2026 11:51:24 GMT  |
-----------------------------------------------------------------------------------------------------



In [46]:
customer_csv = session.read \
    .option("format_name", "AVD.TEST.CSV_FORMAT") \
    .csv("@AVD.TEST.STAGES/customer.csv")

customer_csv.show()

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"c1"   |"c2"                |"c3"                                     |"c4"  |"c5"             |"c6"     |"c7"        |"c8"                                                |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|60001  |Customer#000060001  |9Ii4zQn9cX                               |14    |24-678-784-9652  |9957.56  |HOUSEHOLD   |l theodolites boost slyly at the platelets: per...  |
|60002  |Customer#000060002  |ThGBMjDwKzkoOxhz                         |15    |25-782-500-8435  |742.46   |BUILDING    | beans. fluffily regular packages                   |
|60003  |Customer#000060003  |Ed hbPtTXMTAsgGhCr4HuTzK,Md2             |16    |26-859-847-7640  |2526.92  |BUILDING    |fully pend

In [55]:
customer_csv = session.read \
    .option("format_name", "AVD.TEST.CSV_FORMAT") \
    .option("PARSE_HEADER", True) \
    .csv("@AVD.TEST.STAGES/customer.csv")
customer_csv.show()


-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"c1"   |"c2"                |"c3"                                     |"c4"  |"c5"             |"c6"     |"c7"        |"c8"                                                |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|60001  |Customer#000060001  |9Ii4zQn9cX                               |14    |24-678-784-9652  |9957.56  |HOUSEHOLD   |l theodolites boost slyly at the platelets: per...  |
|60002  |Customer#000060002  |ThGBMjDwKzkoOxhz                         |15    |25-782-500-8435  |742.46   |BUILDING    | beans. fluffily regular packages                   |
|60003  |Customer#000060003  |Ed hbPtTXMTAsgGhCr4HuTzK,Md2             |16    |26-859-847-7640  |2526.92  |BUILDING    |fully pend